In [1]:
import json
from pathlib import Path
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_PATH = Path("../data/qwen_train_data.jsonl")

/Users/sasha/miniconda3/envs/lima/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [3]:
def load_jsonl(path):
    examples = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            examples.append(json.loads(line))
    return examples

examples = load_jsonl(DATA_PATH)

print(f"Number of examples: {len(examples)}")
print(f"Available fields: {list(examples[0].keys())}")

Number of examples: 1030
Available fields: ['text']


In [4]:
ex1 = examples[0]["text"]
ex2 = examples[1]["text"]

print(len(ex1))
print(len(ex2))

1828
3920


In [ ]:
enc1 = tokenizer(ex1)     # enc1 is a Hugging Face BatchEncoding object
enc2 = tokenizer(ex2)

print(len(enc1["input_ids"]))     # number of tokens in each example, not the number of characters
print(len(enc2["input_ids"]))

411
822


In [11]:
print(enc1["input_ids"])
print(enc1["attention_mask"])

[151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 6713, 8109, 7761, 3271, 30, 3216, 7203, 358, 3076, 1293, 6010, 11906, 320, 80060, 2845, 2878, 279, 8109, 1172, 568, 151645, 198, 151644, 77091, 198, 785, 3405, 374, 12040, 7205, 323, 825, 1265, 1896, 1119, 2692, 429, 279, 8109, 537, 1172, 17167, 315, 33213, 11, 714, 1083, 2770, 530, 7761, 320, 23362, 533, 7761, 8, 323, 855, 1448, 275, 14212, 78302, 19101, 7761, 13, 23405, 11, 438, 9023, 12357, 1331, 12295, 1671, 614, 16317, 11, 46906, 6430, 374, 1602, 2989, 11, 438, 279, 11220, 43381, 14011, 8109, 374, 1602, 2155, 504, 279, 6683, 8109, 624, 11209, 11, 1283, 274, 17680, 1526, 5257, 27985, 11, 279, 4226, 311, 279, 3405, 374, 3520, 48623, 4285, 25, 7414, 11, 8109, 7761, 44566, 624, 641, 220, 279, 6683, 8109, 2770, 530, 7761, 44566, 304, 279, 8109, 320, 42, 43183, 3096, 83, 11, 220, 17, 15, 15, 24, 568, 8280, 530, 7761, 525, 6398, 304, 264, 51809, 31

In [8]:
batch = tokenizer(
    [ex1, ex2],
    padding=True,
    return_tensors="pt"     # converts the results into PyTorch tensors instead of ordinary Python lists
)

In [12]:
batch["input_ids"].shape

torch.Size([2, 822])

In [13]:
print(batch["attention_mask"])

tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]])


In [14]:
print(batch["input_ids"])

tensor([[151644,   8948,    198,  ..., 151643, 151643, 151643],
        [151644,   8948,    198,  ...,     13, 151645,    198]])


In [15]:
tokenizer.pad_token
tokenizer.pad_token_id

151643

In [16]:
pad_id = tokenizer.pad_token_id
print(pad_id)
print(tokenizer.decode([pad_id]))

151643
<|endoftext|>


In [17]:
for token_id, mask in zip(
    batch["input_ids"][0][-30:],
    batch["attention_mask"][0][-30:]
):
    print(token_id.item(), mask.item(), repr(tokenizer.decode([token_id])))

151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'
151643 0 '<|endoftext|>'


In [ ]:
labels = batch["input_ids"].clone()
labels[batch["attention_mask"] == 0] = -100     # don't compute loss for this position

print(labels[0][-30:])
print(labels[1][-30:])

tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100])
tensor([   264,  14977,  19415,    369,    432,     13,   4695,    264,   1661,
          2310,  10008,     12,  92350,  17654,   1075,    856,     23,     21,
            62,     21,     19,     11,    429,    594,   2494,    582,   3535,
            13, 151645,    198])


In [41]:
for i in range(2):
    seq_len = int(batch["attention_mask"][i].sum())
    ids = batch["input_ids"][i]
    print(f"\nExample {i}:")
    for j in range(seq_len - 10, seq_len):
        token_id = ids[j].item()
        print(j, token_id, repr(tokenizer.decode([token_id])))


Example 0:
401 303 'nd'
402 1578 ' ed'
403 11 ','
404 4182 ' Ne'
405 324 'ur'
406 24202 'onal'
407 21248 ' Migration'
408 568 ').'
409 151645 '<|im_end|>'
410 198 '\n'

Example 1:
812 19 '4'
813 11 ','
814 429 ' that'
815 594 "'s"
816 2494 ' something'
817 582 ' we'
818 3535 ' understand'
819 13 '.'
820 151645 '<|im_end|>'
821 198 '\n'
